In [ ]:
!pip install kagglehub --quiet

import kagglehub
import os
import zipfile
import shutil
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import json
import glob

# ==============================================================================
# MOUNT GOOGLE DRIVE
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục project trên Drive
PROJECT_DIR = "/content/drive/MyDrive/PlantDiseaseProject"
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"✓ Project directory: {PROJECT_DIR}")

# ==============================================================================
# CẤU HÌNH GPU T4 TỐI ƯU
# ==============================================================================
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU configured: {len(gpus)} GPU(s) available")
        print(f"✓ GPU name: {tf.test.gpu_device_name()}")
    except RuntimeError as e:
        print(e)

# Mixed precision để tăng tốc trên T4
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print('✓ Mixed precision enabled: Compute dtype=%s, variable dtype=%s' % 
      (policy.compute_dtype, policy.variable_dtype))

print("GPU:", tf.config.list_physical_devices('GPU'))

# ==============================================================================
# DOWNLOAD DATASET
# ==============================================================================
path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print("Dataset path:", path)

# Xử lý nếu là file zip
if os.path.isfile(path) and path.endswith(".zip"):
    extract_dir = path.replace(".zip", "")
    with zipfile.ZipFile(path, 'r') as z:
        z.extractall(extract_dir)
    base_dir = extract_dir
else:
    base_dir = path

data_dir = os.path.join(base_dir, "plantvillage dataset", "color")
print("Using directory:", data_dir)

# ==============================================================================
# CHỌN 5 LOẠI CÂY PHỔ BIẾN Ở VIỆT NAM (2-5 classes mỗi loại)
# ==============================================================================
# Chọn các loại cây có ít classes để training nhanh
VIETNAM_CROPS = [
    "Potato",           # Khoai tây - 3 classes
    "Corn_(maize)",     # Ngô - 4 classes
    "Strawberry",       # Dâu tây - 2 classes
    "Grape",            # Nho - 4 classes
    "Peach",            # Đào - 2 classes
]

print("\n" + "="*60)
print("CHỌN 5 LOẠI CÂY PHỔ BIẾN Ở VIỆT NAM (tránh Tomato)")
print("="*60)

all_classes = os.listdir(data_dir)
selected_classes = []

for crop in VIETNAM_CROPS:
    matching = [c for c in all_classes if crop in c]
    selected_classes.extend(matching)
    if matching:
        print(f"✓ {crop}: {len(matching)} classes")

print(f"\nTổng số classes: {len(selected_classes)}")
print("="*60)

# Tạo thư mục filtered dataset
filtered_dir = "/content/PlantVillageFiltered"
os.makedirs(filtered_dir, exist_ok=True)

for cls in selected_classes:
    src = os.path.join(data_dir, cls)
    dst = os.path.join(filtered_dir, cls)
    if not os.path.exists(dst):
        shutil.copytree(src, dst)

data_dir = filtered_dir
print(f"\n✓ Đã filter dataset: {len(os.listdir(data_dir))} classes")

# ==============================================================================
# SPLIT DATA
# ==============================================================================
split_dir = "/content/PlantVillageSplit"
train_dir = os.path.join(split_dir, "train")
val_dir = os.path.join(split_dir, "val")

if not os.path.exists(split_dir):
    class_names = os.listdir(data_dir)

    for cls in class_names:
        cls_path = os.path.join(data_dir, cls)
        imgs = os.listdir(cls_path)

        train_files, val_files = train_test_split(
            imgs, test_size=0.2, random_state=42
        )

        os.makedirs(f"{train_dir}/{cls}", exist_ok=True)
        os.makedirs(f"{val_dir}/{cls}", exist_ok=True)

        for f in train_files:
            shutil.copy(os.path.join(cls_path, f), f"{train_dir}/{cls}")

        for f in val_files:
            shutil.copy(os.path.join(cls_path, f), f"{val_dir}/{cls}")

    print("✓ Dataset đã được split!")
else:
    print("✓ Dataset đã split từ trước")

# ==============================================================================
# DATA PREPARATION
# ==============================================================================
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

img_size = (224, 224)
batch_size = 64  # Tăng batch size cho T4

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir, 
    target_size=img_size, 
    batch_size=batch_size, 
    class_mode='categorical',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    val_dir, 
    target_size=img_size, 
    batch_size=batch_size, 
    class_mode='categorical',
    shuffle=False
)

num_classes = train_gen.num_classes
print(f"\n✓ Số classes: {num_classes}")
print(f"✓ Training samples: {train_gen.samples}")
print(f"✓ Validation samples: {val_gen.samples}")

# Tính class weights
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
class_weights_dict = dict(enumerate(class_weights))

# ==============================================================================
# MODEL BUILDING
# ==============================================================================
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam

def build_fast_model(num_classes):
    base = ResNet50(
        weights="imagenet", 
        include_top=False, 
        input_shape=(224, 224, 3)
    )
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = BatchNormalization()(x)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)
    out = Dense(num_classes, activation="softmax", dtype='float32')(x)

    model = Model(inputs=base.input, outputs=out)
    
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    return model, base

# ==============================================================================
# RESUME TRAINING LOGIC
# ==============================================================================
def save_training_state(phase, epoch, history, filepath):
    """Lưu trạng thái training"""
    state = {
        'phase': phase,
        'epoch': epoch,
        'history': history
    }
    with open(filepath, 'w') as f:
        json.dump(state, f)
    print(f"✓ Saved training state: Phase {phase}, Epoch {epoch}")

def load_training_state(filepath):
    """Load trạng thái training"""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            state = json.load(f)
        print(f"✓ Loaded training state: Phase {state['phase']}, Epoch {state['epoch']}")
        return state
    return None

# Đường dẫn lưu trữ
MODEL_PATH = os.path.join(PROJECT_DIR, "model_checkpoint.h5")
STATE_PATH = os.path.join(PROJECT_DIR, "training_state.json")
BEST_MODEL_PATH = os.path.join(PROJECT_DIR, "best_model.h5")

# Kiểm tra xem có checkpoint không
training_state = load_training_state(STATE_PATH)

if training_state:
    print("\n" + "="*60)
    print("PHÁT HIỆN CHECKPOINT - RESUME TRAINING")
    print("="*60)
    model = load_model(MODEL_PATH)
    base_model = model.layers[0]
    start_phase = training_state['phase']
    history_1 = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}
    history_2 = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}
    
    if start_phase == 1:
        for key in history_1.keys():
            if key in training_state['history']:
                history_1[key] = training_state['history'][key]
    elif start_phase == 2:
        # Load cả history phase 1
        history_1 = training_state['history'].get('phase1_history', history_1)
        for key in history_2.keys():
            if key in training_state['history']:
                history_2[key] = training_state['history'][key]
    
    print(f"✓ Resume từ Phase {start_phase}")
else:
    print("\n✓ Bắt đầu training mới")
    model, base_model = build_fast_model(num_classes)
    start_phase = 1
    history_1 = None
    history_2 = None

print("\n✓ Model ready")

# ==============================================================================
# CUSTOM CALLBACK - Lưu sau mỗi epoch
# ==============================================================================
from tensorflow.keras.callbacks import Callback

class CheckpointCallback(Callback):
    def __init__(self, model_path, state_path, phase, prev_history=None):
        super().__init__()
        self.model_path = model_path
        self.state_path = state_path
        self.phase = phase
        self.prev_history = prev_history or {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}
        
    def on_epoch_end(self, epoch, logs=None):
        # Lưu model
        self.model.save(self.model_path)
        
        # Cập nhật history
        current_history = {
            'accuracy': self.prev_history['accuracy'] + self.model.history.history['accuracy'],
            'val_accuracy': self.prev_history['val_accuracy'] + self.model.history.history['val_accuracy'],
            'loss': self.prev_history['loss'] + self.model.history.history['loss'],
            'val_loss': self.prev_history['val_loss'] + self.model.history.history['val_loss']
        }
        
        # Lưu state
        save_training_state(self.phase, epoch + 1, current_history, self.state_path)

# ==============================================================================
# TRAINING - PHASE 1
# ==============================================================================
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import time

start_time = time.time()

if start_phase <= 1:
    print("\n" + "="*60)
    print("PHASE 1: Training head only")
    print("="*60)
    
    # Callbacks
    early = EarlyStopping(
        monitor="val_accuracy",
        patience=8,
        restore_best_weights=True,
        mode='max',
        verbose=1
    )
    
    best_checkpoint = ModelCheckpoint(
        BEST_MODEL_PATH, 
        save_best_only=True, 
        monitor='val_accuracy',
        mode='max',
        verbose=1
    )
    
    reduce_lr = ReduceLROnPlateau(
        monitor='val_accuracy', 
        factor=0.5, 
        patience=3,
        min_lr=1e-6, 
        verbose=1,
        mode='max'
    )
    
    checkpoint_saver = CheckpointCallback(MODEL_PATH, STATE_PATH, phase=1, prev_history=history_1)
    
    history_1 = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=15,
        class_weight=class_weights_dict,
        callbacks=[early, best_checkpoint, reduce_lr, checkpoint_saver],
        verbose=1
    )
    
    phase1_time = time.time() - start_time
    print(f"✓ Phase 1 hoàn thành trong {phase1_time/60:.1f} phút")
    
    # Lưu history phase 1
    phase1_history = {
        'accuracy': history_1.history['accuracy'],
        'val_accuracy': history_1.history['val_accuracy'],
        'loss': history_1.history['loss'],
        'val_loss': history_1.history['val_loss']
    }
    
    # Update state để chuyển sang phase 2
    save_training_state(2, 0, {'phase1_history': phase1_history}, STATE_PATH)
    model.save(MODEL_PATH)

# ==============================================================================
# TRAINING - PHASE 2
# ==============================================================================
if start_phase <= 2:
    print("\n" + "="*60)
    print("PHASE 2: Fine-tune conv5_block")
    print("="*60)
    
    # Load lại model nếu cần
    if start_phase == 2 and training_state:
        model = load_model(MODEL_PATH)
        base_model = model.layers[0]
    
    # Unfreeze conv5_block
    for layer in base_model.layers:
        if 'conv5_block' in layer.name:
            layer.trainable = True
    
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    print(f"Trainable layers: {sum([1 for layer in base_model.layers if layer.trainable])}")
    
    # Callbacks
    early = EarlyStopping(
        monitor="val_accuracy",
        patience=8,
        restore_best_weights=True,
        mode='max',
        verbose=1
    )
    
    best_checkpoint = ModelCheckpoint(
        BEST_MODEL_PATH, 
        save_best_only=True, 
        monitor='val_accuracy',
        mode='max',
        verbose=1
    )
    
    reduce_lr = ReduceLROnPlateau(
        monitor='val_accuracy', 
        factor=0.5, 
        patience=3,
        min_lr=1e-6, 
        verbose=1,
        mode='max'
    )
    
    # Load history từ phase 1
    if training_state and 'phase1_history' in training_state['history']:
        prev_history = training_state['history']['phase1_history']
    elif history_1:
        prev_history = {
            'accuracy': history_1.history['accuracy'],
            'val_accuracy': history_1.history['val_accuracy'],
            'loss': history_1.history['loss'],
            'val_loss': history_1.history['val_loss']
        }
    else:
        prev_history = None
    
    checkpoint_saver = CheckpointCallback(MODEL_PATH, STATE_PATH, phase=2, prev_history=history_2)
    
    history_2 = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=15,
        class_weight=class_weights_dict,
        callbacks=[early, best_checkpoint, reduce_lr, checkpoint_saver],
        verbose=1
    )

total_time = time.time() - start_time
print(f"\n✓ TRAINING HOÀN THÀNH trong {total_time/60:.1f} phút")

# ==============================================================================
# EVALUATION
# ==============================================================================
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load best model
model.load_weights(BEST_MODEL_PATH)

print("\n⏳ Đang đánh giá model...")
y_true = val_gen.classes
y_pred_probs = model.predict(val_gen, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

test_acc = np.mean(y_pred == y_true)

print("\n" + "="*60)
print("KẾT QUẢ CUỐI CÙNG")
print("="*60)
print(f"✓ Validation Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
if test_acc >= 0.90:
    print(f"✓ ĐẠT MỤC TIÊU! Accuracy ≥ 0.90")
else:
    print(f"✗ Cần cải thiện thêm {(0.90-test_acc)*100:.2f}%")
print(f"✓ Tổng thời gian: {total_time/60:.1f} phút")
print("="*60)

# Classification report
classes = list(val_gen.class_indices.keys())
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=classes, digits=4))

# ==============================================================================
# VẼ BIỂU ĐỒ VÀ LƯU VÀO DRIVE
# ==============================================================================
print("\n⏳ Đang vẽ biểu đồ...")

# Combine histories
if history_1 and history_2:
    all_acc = history_1.history['accuracy'] + history_2.history['accuracy']
    all_val_acc = history_1.history['val_accuracy'] + history_2.history['val_accuracy']
    all_loss = history_1.history['loss'] + history_2.history['loss']
    all_val_loss = history_1.history['val_loss'] + history_2.history['val_loss']
elif history_1:
    all_acc = history_1.history['accuracy']
    all_val_acc = history_1.history['val_accuracy']
    all_loss = history_1.history['loss']
    all_val_loss = history_1.history['val_loss']
else:
    all_acc = history_2.history['accuracy']
    all_val_acc = history_2.history['val_accuracy']
    all_loss = history_2.history['loss']
    all_val_loss = history_2.history['val_loss']

# Figure 1: Training History
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(all_acc, label='Train Accuracy', linewidth=2, marker='o', markersize=3)
plt.plot(all_val_acc, label='Val Accuracy', linewidth=2, marker='s', markersize=3)
plt.axhline(y=0.90, color='red', linestyle='--', linewidth=2, label='Target 0.90')
plt.title('Accuracy Over Time', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(all_loss, label='Train Loss', linewidth=2, marker='o', markersize=3)
plt.plot(all_val_loss, label='Val Loss', linewidth=2, marker='s', markersize=3)
plt.title('Loss Over Time', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
if history_1 and history_2:
    lr_timeline = [1e-3]*len(history_1.history['loss']) + [1e-4]*len(history_2.history['loss'])
elif history_1:
    lr_timeline = [1e-3]*len(history_1.history['loss'])
else:
    lr_timeline = [1e-4]*len(history_2.history['loss'])
    
epochs_range = range(1, len(lr_timeline)+1)
plt.plot(epochs_range, lr_timeline, linewidth=2, color='green', marker='o', markersize=4)
plt.title('Learning Rate Schedule', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.yscale('log')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'training_history.png'), dpi=100, bbox_inches='tight')
plt.show()

# Figure 2: Confusion Matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[c.split('___')[1] if '___' in c else c for c in classes],
            yticklabels=[c.split('___')[1] if '___' in c else c for c in classes],
            cbar_kws={'label': 'Count'})
plt.title(f'Confusion Matrix\nAccuracy: {test_acc:.4f}', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'confusion_matrix.png'), dpi=100, bbox_inches='tight')
plt.show()

# Figure 3: Per-class accuracy
from collections import defaultdict
class_correct = defaultdict(int)
class_total = defaultdict(int)

for true_label, pred_label in zip(y_true, y_pred):
    class_total[true_label] += 1
    if true_label == pred_label:
        class_correct[true_label] += 1

class_accuracies = []
for class_idx in sorted(class_total.keys()):
    acc = class_correct[class_idx] / class_total[class_idx]
    class_name = classes[class_idx].split('___')[1] if '___' in classes[class_idx] else classes[class_idx]
    class_accuracies.append((class_name, acc))

class_accuracies.sort(key=lambda x: x[1])

plt.figure(figsize=(12, 6))
names, accs = zip(*class_accuracies)
colors = ['red' if acc < 0.85 else 'orange' if acc < 0.90 else 'green' for acc in accs]
bars = plt.barh(names, accs, color=colors, alpha=0.7, edgecolor='black')

plt.axvline(x=0.90, color='red', linestyle='--', linewidth=2, label='Target 0.90')
plt.xlabel('Accuracy', fontsize=12)
plt.ylabel('Class', fontsize=12)
plt.title('Per-Class Accuracy', fontsize=14, fontweight='bold')
plt.xlim(0, 1.05)
plt.legend()
plt.grid(True, axis='x', alpha=0.3)

for i, (bar, acc) in enumerate(zip(bars, accs)):
    plt.text(acc + 0.01, i, f'{acc:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'per_class_accuracy.png'), dpi=100, bbox_inches='tight')
plt.show()

# ==============================================================================
# LƯU MODEL CUỐI CÙNG
# ==============================================================================
final_model_path = os.path.join(PROJECT_DIR, 'final_model.h5')
model.save(final_model_path)

# Xóa checkpoint files
if os.path.exists(MODEL_PATH):
    os.remove(MODEL_PATH)
if os.path.exists(STATE_PATH):
    os.remove(STATE_PATH)

print("\n" + "="*60)
print("📊 TÓM TẮT")
print("="*60)
print(f"✓ Số classes: {num_classes}")
print(f"✓ Training samples: {train_gen.samples}")
print(f"✓ Validation samples: {val_gen.samples}")
print(f"✓ Validation Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"✓ Tổng thời gian: {total_time/60:.1f} phút")
print(f"✓ Model đã lưu tại: {final_model_path}")
print(f"✓ Biểu đồ đã lưu tại: {PROJECT_DIR}")
print("="*60)

# ==============================================================================
# BIỂU ĐỒ SO SÁNH NÂNG CAO
# ==============================================================================
print("\n⏳ Đang tạo biểu đồ so sánh...")

# Figure 4: Dashboard tổng hợp
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. So sánh Phase 1 vs Phase 2
ax1 = fig.add_subplot(gs[0, :2])
if history_1 and history_2:
    phase1_epochs = len(history_1.history['val_accuracy'])
    phase2_epochs = len(history_2.history['val_accuracy'])
    
    epochs_p1 = range(1, phase1_epochs + 1)
    epochs_p2 = range(phase1_epochs + 1, phase1_epochs + phase2_epochs + 1)
    
    ax1.plot(epochs_p1, history_1.history['val_accuracy'], 'o-', 
             linewidth=2, markersize=6, label='Phase 1 (Frozen)', color='blue')
    ax1.plot(epochs_p2, history_2.history['val_accuracy'], 's-', 
             linewidth=2, markersize=6, label='Phase 2 (Fine-tune)', color='red')
    
    ax1.axvline(x=phase1_epochs, color='gray', linestyle='--', linewidth=2, alpha=0.5)
    ax1.text(phase1_epochs, 0.5, 'Phase Transition', rotation=90, 
             verticalalignment='center', fontsize=10)
    
    ax1.axhline(y=0.90, color='green', linestyle='--', linewidth=2, label='Target 0.90')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Validation Accuracy', fontsize=12)
    ax1.set_title('Phase 1 vs Phase 2 Performance', fontsize=14, fontweight='bold')
    ax1.legend(loc='lower right')
    ax1.grid(True, alpha=0.3)

# 2. Accuracy improvement bar chart
ax2 = fig.add_subplot(gs[0, 2])
if history_1 and history_2:
    phase1_best = max(history_1.history['val_accuracy'])
    phase2_best = max(history_2.history['val_accuracy'])
    improvement = (phase2_best - phase1_best) * 100
    
    bars = ax2.bar(['Phase 1\nBest', 'Phase 2\nBest', 'Improvement'], 
                   [phase1_best*100, phase2_best*100, improvement],
                   color=['blue', 'red', 'green'], alpha=0.7, edgecolor='black')
    
    for bar in bars[:2]:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Improvement annotation
    ax2.text(bars[2].get_x() + bars[2].get_width()/2., improvement/2,
            f'+{improvement:.2f}%', ha='center', va='center', 
            fontsize=11, fontweight='bold', color='white')
    
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training Improvement', fontsize=12, fontweight='bold')
    ax2.grid(True, axis='y', alpha=0.3)

# 3. Per-crop performance comparison
ax3 = fig.add_subplot(gs[1, :])
# Group classes by crop
crop_performance = {}
for class_name, acc in class_accuracies:
    # Extract crop name (before first space or underscore)
    for crop in VIETNAM_CROPS:
        crop_clean = crop.replace('_(maize)', '').replace(',_bell', '')
        if crop_clean.lower() in class_name.lower():
            if crop not in crop_performance:
                crop_performance[crop] = []
            crop_performance[crop].append(acc)
            break

crop_names = list(crop_performance.keys())
crop_avg_acc = [np.mean(crop_performance[crop]) * 100 for crop in crop_names]
crop_min_acc = [np.min(crop_performance[crop]) * 100 for crop in crop_names]
crop_max_acc = [np.max(crop_performance[crop]) * 100 for crop in crop_names]
crop_std = [np.std(crop_performance[crop]) * 100 for crop in crop_names]

x_pos = np.arange(len(crop_names))
colors_crop = ['green' if avg >= 90 else 'orange' if avg >= 85 else 'red' 
               for avg in crop_avg_acc]

bars = ax3.bar(x_pos, crop_avg_acc, color=colors_crop, alpha=0.7, edgecolor='black', linewidth=2)
ax3.errorbar(x_pos, crop_avg_acc, yerr=crop_std, fmt='none', 
             ecolor='black', capsize=5, capthick=2)

for i, (bar, avg, min_v, max_v) in enumerate(zip(bars, crop_avg_acc, crop_min_acc, crop_max_acc)):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{avg:.1f}%\n({min_v:.1f}-{max_v:.1f})', 
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax3.axhline(y=90, color='red', linestyle='--', linewidth=2, label='Target 90%')
ax3.set_xticks(x_pos)
ax3.set_xticklabels([c.replace('_', ' ').replace('(maize)', '') for c in crop_names], 
                     rotation=15, ha='right')
ax3.set_ylabel('Average Accuracy (%)', fontsize=12)
ax3.set_title('Performance by Crop Type (with min-max range)', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(True, axis='y', alpha=0.3)
ax3.set_ylim(0, 105)

# 4. Healthy vs Disease comparison
ax4 = fig.add_subplot(gs[2, 0])
healthy_acc = []
disease_acc = []

for class_name, acc in class_accuracies:
    if 'healthy' in class_name.lower():
        healthy_acc.append(acc)
    else:
        disease_acc.append(acc)

if healthy_acc and disease_acc:
    data_to_plot = [np.array(healthy_acc)*100, np.array(disease_acc)*100]
    bp = ax4.boxplot(data_to_plot, labels=['Healthy', 'Disease'], 
                     patch_artist=True, showmeans=True,
                     meanprops=dict(marker='D', markerfacecolor='red', markersize=8))
    
    for patch, color in zip(bp['boxes'], ['lightgreen', 'lightcoral']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax4.axhline(y=90, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax4.set_ylabel('Accuracy (%)', fontsize=12)
    ax4.set_title('Healthy vs Disease Classes', fontsize=12, fontweight='bold')
    ax4.grid(True, axis='y', alpha=0.3)

# 5. Training time breakdown
ax5 = fig.add_subplot(gs[2, 1])
if history_1 and history_2:
    time_data = {
        'Setup': 5,
        'Phase 1': phase1_time/60,
        'Phase 2': (total_time - phase1_time)/60,
    }
    
    colors_time = ['gray', 'blue', 'red']
    wedges, texts, autotexts = ax5.pie(time_data.values(), labels=time_data.keys(), 
                                        autopct='%1.1f%%', colors=colors_time, 
                                        startangle=90, textprops={'fontsize': 10})
    
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
    
    ax5.set_title(f'Training Time Breakdown\nTotal: {total_time/60:.1f} min', 
                  fontsize=12, fontweight='bold')

# 6. Confusion intensity (most confused pairs)
ax6 = fig.add_subplot(gs[2, 2])
# Find most confused class pairs
confusion_pairs = []
for i in range(len(cm)):
    for j in range(len(cm)):
        if i != j and cm[i][j] > 0:
            confusion_pairs.append((classes[i], classes[j], cm[i][j]))

confusion_pairs.sort(key=lambda x: x[2], reverse=True)
top_confusions = confusion_pairs[:5]

if top_confusions:
    labels = [f"{c[0].split('___')[1][:15] if '___' in c[0] else c[0][:15]}\n→\n{c[1].split('___')[1][:15] if '___' in c[1] else c[1][:15]}" 
              for c in top_confusions]
    values = [c[2] for c in top_confusions]
    
    bars = ax6.barh(labels, values, color='coral', alpha=0.7, edgecolor='black')
    
    for bar, val in zip(bars, values):
        ax6.text(val + 0.5, bar.get_y() + bar.get_height()/2., 
                f'{int(val)}', va='center', fontsize=10, fontweight='bold')
    
    ax6.set_xlabel('Confusion Count', fontsize=12)
    ax6.set_title('Top 5 Confused Class Pairs', fontsize=12, fontweight='bold')
    ax6.grid(True, axis='x', alpha=0.3)

plt.suptitle(f'Comprehensive Training Analysis Dashboard\nFinal Accuracy: {test_acc:.4f}', 
             fontsize=16, fontweight='bold', y=0.995)

plt.savefig(os.path.join(PROJECT_DIR, 'comparison_dashboard.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

# ==============================================================================
# SUMMARY TABLE
# ==============================================================================
print("\n" + "="*60)
print("📊 BẢNG SO SÁNH CHI TIẾT")
print("="*60)

if history_1 and history_2:
    print("\n🔵 PHASE 1 (Frozen ResNet50):")
    print(f"  • Best Val Accuracy: {max(history_1.history['val_accuracy']):.4f}")
    print(f"  • Final Val Accuracy: {history_1.history['val_accuracy'][-1]:.4f}")
    print(f"  • Epochs trained: {len(history_1.history['val_accuracy'])}")
    print(f"  • Time: {phase1_time/60:.1f} minutes")
    
    print("\n🔴 PHASE 2 (Fine-tuned conv5_block):")
    print(f"  • Best Val Accuracy: {max(history_2.history['val_accuracy']):.4f}")
    print(f"  • Final Val Accuracy: {history_2.history['val_accuracy'][-1]:.4f}")
    print(f"  • Epochs trained: {len(history_2.history['val_accuracy'])}")
    print(f"  • Time: {(total_time - phase1_time)/60:.1f} minutes")
    
    improvement = (max(history_2.history['val_accuracy']) - max(history_1.history['val_accuracy'])) * 100
    print(f"\n✨ IMPROVEMENT: +{improvement:.2f}%")

print("\n🌿 PERFORMANCE BY CROP:")
for crop in crop_names:
    avg = np.mean(crop_performance[crop]) * 100
    std = np.std(crop_performance[crop]) * 100
    min_v = np.min(crop_performance[crop]) * 100
    max_v = np.max(crop_performance[crop]) * 100
    status = "✓" if avg >= 90 else "⚠" if avg >= 85 else "✗"
    print(f"  {status} {crop:20s}: {avg:5.2f}% (±{std:.2f}%) [{min_v:.1f}-{max_v:.1f}%]")

if healthy_acc and disease_acc:
    print("\n🏥 HEALTHY vs DISEASE:")
    print(f"  • Healthy classes avg: {np.mean(healthy_acc)*100:.2f}%")
    print(f"  • Disease classes avg: {np.mean(disease_acc)*100:.2f}%")
    diff = (np.mean(healthy_acc) - np.mean(disease_acc)) * 100
    if diff > 0:
        print(f"  • Healthy performs better by: {diff:.2f}%")
    else:
        print(f"  • Disease performs better by: {abs(diff):.2f}%")

print("\n" + "="*60)
print(f"✓ Dashboard đã lưu: {os.path.join(PROJECT_DIR, 'comparison_dashboard.png')}")
print("\n🎉 HOÀN THÀNH!")

ValueError: mount failed